In [1]:
import torch
import sys
import pandas as pd
sys.path.append('../')
from utilities import load_embedding
from torch.utils.data import Dataset, DataLoader
from utilities import print_exams
import torch.nn.functional as F

In [2]:
class fluProfiler_Dataset(Dataset):
    def __init__(self, DataFrame):
        self.emb_file_name_a = ('matrix_' + DataFrame['seq_id_a']).tolist()
        self.emb_file_name_b = ('matrix_' + DataFrame['seq_id_b']).tolist()
        self.emb_file_name_c = ('matrix_' + DataFrame['seq_id_c']).tolist()
        self.emb_file_name_d = ('matrix_' + DataFrame['seq_id_d']).tolist()

        self.strainPassCats = convert_Pass2tensor(('<cls>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat'] + '<eos>').tolist())

        self.labels = torch.tensor(DataFrame['label'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.emb_file_name_a[idx], self.emb_file_name_b[idx], self.emb_file_name_c[idx], self.emb_file_name_d[idx], \
               self.strainPassCats[idx], self.labels[idx]

def convert_Pass2tensor(pass_cats):
    result = [
        item.replace('<cls>', '0').replace('<eos>', '1').replace('<EGG>', '2').replace('<CELL>', '3').replace('<BOTH>', '4')
        for item in pass_cats
    ]
    result = torch.tensor([[int(number) for number in [char for char in item]] for item in result])
    return result

def generate_matrix(matrix_list):
    seq_len = [mat.shape[0] for mat in matrix_list]
    max_len = max(seq_len)
    mask_list = []
    for i in range(len(matrix_list)): 
        matrix_list[i] = F.pad(matrix_list[i], (0, 0, 0, max_len - seq_len[i]))
        mask = torch.concat((torch.ones(1,seq_len[i]),torch.zeros(1,max_len-seq_len[i])),axis=1)
        mask_list.append(mask)
    matrix = torch.stack(matrix_list)
    mask = torch.stack(mask_list).view(len(matrix_list),max_len)
    return matrix, mask


In [4]:
device = torch.device('cuda:0')
model = torch.load('../../trained_model/fluProfiler_ultra_A40/2025-09-12_19-51-43.pth', 
weights_only=False,
map_location=device)

In [6]:
test_data = pd.read_csv('../../data/data_40/fluProfiler_ultra/test.csv')

In [7]:
sequence_names = pd.concat([test_data['seq_id_a'], test_data['seq_id_b'],
                            test_data['seq_id_c'], test_data['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
emb_dict_Crick = load_embedding("../../data/data_40/fluProfiler_ultra/embedding", files=sequence_names)

In [9]:
test_dataloader = DataLoader(fluProfiler_Dataset(test_data), batch_size=100, shuffle=False)

In [14]:
prediction_ls_test = []
reference_ls_test = []
logits_ls = []
loss_ls_test = []
model.eval()
for batch in test_dataloader:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch

    matrixs_a, masks_a = generate_matrix([emb_dict_Crick[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict_Crick[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict_Crick[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict_Crick[key] for key in emb_file_name_d])

    matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c,
                                        matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b,
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats,
                                        labels=labels)

    loss_ls_test.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls_test.extend(output.view(-1).tolist())
    reference_ls_test.extend(labels.tolist())

test_mae, test_mse, test_pearson, test_spearman, test_R2 = print_exams(reference_ls_test, prediction_ls_test)

MAE: 0.61326
MSE: 0.64083
pearson correlation: 0.91670
spearman correlation: 0.88613
R2_score: 0.81506


In [8]:
test_data.to_csv('./ultra_test_result.csv', index=False)

In [26]:
exam_df = test_data[(test_data['tag'] == '41') | (test_data['tag'] == '42') | (test_data['tag'] == '43')]
print_exams(exam_df['label'], exam_df['prediction'])

MAE: 0.51431
MSE: 0.44762
pearson correlation: 0.87046
spearman correlation: 0.81020
R2_score: 0.69907


(0.5143057912891895,
 0.4476246996535247,
 PearsonRResult(statistic=0.8704598413588309, pvalue=0.0),
 SignificanceResult(statistic=0.8102030020540709, pvalue=4.701777510791371e-269),
 0.6990702385402222)